# Gemini Stage Explorer

Explore cleaned Gemini OCR/classification outputs with the same filters used in the pipeline, plus handy summaries of flag categories per attribute.

## Quick start

1. Activate the pipeline's Python environment.
3. Run the notebook cells from top to bottom. Choose the Gemini output JSONL you want to review (base run or a filtered run) and interact with the charts.


In [ ]:
import html
import json
from pathlib import Path

import yaml

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install pandas to use this notebook (pip install pandas)."
    ) from exc

try:
    import plotly.express as px
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install plotly to use this notebook (pip install plotly)."
    ) from exc

try:
    import ipywidgets as widgets
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install ipywidgets to use this notebook (pip install ipywidgets)."
    ) from exc

from IPython.display import Markdown, display, Image as IPImage

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_colwidth", 160)
px.defaults.template = "plotly_white"


In [30]:
def find_project_root(markers=("pyproject.toml", ".git")):
    start = Path.cwd()
    for candidate in [start, *start.parents]:
        if any((candidate / marker).exists() for marker in markers):
            return candidate
    return start

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

CUSTOM_CONFIG_PATH = None  # Override with Path("/custom/config/config.yaml") if needed
config_path = Path(CUSTOM_CONFIG_PATH) if CUSTOM_CONFIG_PATH else PROJECT_ROOT / "config/config.yaml"
if not config_path.exists():
    raise FileNotFoundError(
        f"Could not find a config file next to this notebook. Expected {config_path}."
    )
print(f"Loading config from: {config_path}")

with config_path.open() as f:
    config_data = yaml.safe_load(f) or {}


def resolve_path(path):
    if path is None:
        return None
    candidate = Path(path)
    if not candidate.is_absolute():
        candidate = PROJECT_ROOT / candidate
    return candidate


def load_jsonl(path):
    records = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def discover_outputs(stage_name, filenames):
    outputs = {}
    seen = set()
    ordered_names = [str(name) for name in filenames if name]
    for name in ordered_names:
        candidate = resolve_path(name)
        if candidate and candidate.exists():
            label = f"default run: {name}"
            outputs[label] = candidate
            seen.add(candidate.resolve())
    stage_dir = PROJECT_ROOT / "filtered" / stage_name
    if stage_dir.exists():
        for option in sorted(stage_dir.glob("*")):
            if not option.is_dir():
                continue
            for name in ordered_names:
                candidate = option / name
                if candidate.exists() and candidate.resolve() not in seen:
                    label = f"output/filtered/{stage_name}/{option.name} -> {name}"
                    outputs[label] = candidate
                    seen.add(candidate.resolve())
    return outputs


def build_scene_image_map(index_path, scene_level):
    mapping = {}
    if not index_path or not index_path.exists():
        return mapping
    for record in load_jsonl(index_path):
        image_path = Path(record.get("image_path", ""))
        if not image_path:
            continue
        current = image_path
        steps = max(1, int(scene_level))
        for _ in range(steps):
            current = current.parent
        scene_key = str(current)
        mapping.setdefault(scene_key, []).append(str(image_path))
    for scene, paths in mapping.items():
        mapping[scene] = sorted(paths)
    return mapping


dataset_cfg = config_data.get("dataset", {}) if isinstance(config_data, dict) else {}
gemini_cfg = config_data.get("gemini", {}) if isinstance(config_data, dict) else {}
scene_level = int(gemini_cfg.get("scene_directory_level", 1) or 1)
index_path = resolve_path(dataset_cfg.get("output_jsonl"))
scene_image_map = build_scene_image_map(index_path, scene_level) if index_path and index_path.exists() else {}
if scene_image_map:
    print(f"Loaded image index from {index_path} for scene previews.")
else:
    print("Scene image map unavailable; previews will omit images unless the index is present.")


Project root: /Users/bchauhan/Downloads/data-privacy-tool
Loading config from: /Users/bchauhan/Downloads/data-privacy-tool/config/config.yaml
Loaded image index from /Users/bchauhan/Downloads/data-privacy-tool/image_index.jsonl for scene previews.


In [31]:
gemini_cfg = config_data.get("gemini", {}) if isinstance(config_data, dict) else {}
default_output_name = gemini_cfg.get("final_output_jsonl", "gemini_output.jsonl")
candidate_names = {default_output_name, "gemini_output.jsonl", "gemini_output_cleaned.jsonl"}
gemini_sources = discover_outputs("gemini", candidate_names)

if not gemini_sources:
    raise FileNotFoundError(
        "No Gemini output JSONL files were found. Finish the clean step or update candidate_names."
    )

gemini_source_picker = widgets.Dropdown(
    options=[(label, str(path)) for label, path in gemini_sources.items()],
    description="Gemini output",
    layout=widgets.Layout(width="80%"),
)

display(Markdown("**Select the Gemini result set you want to explore:**"))
display(gemini_source_picker)


**Select the Gemini result set you want to explore:**

Dropdown(description='Gemini output', layout=Layout(width='80%'), options=(('default run: gemini_output_cleane…

In [32]:
selected_gemini_path = Path(gemini_source_picker.value)
gemini_records = load_jsonl(selected_gemini_path)

if not gemini_records:
    raise ValueError(f"No records found in {selected_gemini_path}")

scene_rows = []
for record in gemini_records:
    attributes = record.get("attributes", {}) or {}
    categories = [str(cat).strip() for cat in record.get("flag_categories", []) if cat]
    scene_rows.append(
        {
            "scene_path": record.get("scene_path"),
            "response_text": record.get("response_text"),
            "is_flagged": bool(record.get("is_flagged")),
            "flag_categories": categories,
            **attributes,
        }
    )

scenes_df = pd.DataFrame(scene_rows)
if scenes_df.empty:
    scenes_df = pd.DataFrame(columns=["scene_path", "response_text", "is_flagged", "flag_categories"])

if "flag_categories" not in scenes_df.columns:
    scenes_df["flag_categories"] = [[] for _ in range(len(scenes_df))]
scenes_df["flag_categories"] = scenes_df["flag_categories"].apply(lambda cats: cats if isinstance(cats, list) else [])
if "is_flagged" not in scenes_df.columns:
    scenes_df["is_flagged"] = False
scenes_df["is_flagged"] = scenes_df["is_flagged"].fillna(False)

attribute_columns = [
    col
    for col in scenes_df.columns
    if col not in {"scene_path", "response_text", "is_flagged", "flag_categories"}
]
scenes_df["flag_count"] = scenes_df["flag_categories"].apply(len)
all_categories = sorted({cat for cats in scenes_df["flag_categories"] for cat in (cats or [])})

flagged_total = int(scenes_df["is_flagged"].sum()) if not scenes_df.empty else 0
display(
    Markdown(
        f"Loaded **{len(scenes_df):,}** scenes from `{selected_gemini_path}` with **{flagged_total:,}** flagged."
    )
)
if attribute_columns:
    display(Markdown(f"Available attributes: {', '.join(attribute_columns)}"))
else:
    display(Markdown("Scene records do not contain additional attributes; filters will rely on categories only."))

if scenes_df.empty:
    display(Markdown("⚠️ No Gemini records found; charts below will remain empty until results are available."))

scenes_df.head()


Loaded **7** scenes from `/Users/bchauhan/Downloads/data-privacy-tool/gemini_output_cleaned.jsonl` with **2** flagged.

Available attributes: building, scene, date

,scene_path,response_text,is_flagged,flag_categories,building,scene,date,flag_count
0,data/Smith_Hall_121/500180237/Mon_Jul_10_11:45:07_2023,## Combined Text ##\nNo readable text found in the images.,True,[OTHER],Smith_Hall_121,500180237,Mon_Jul_10_11:45:07_2023,1
1,data/Smith_Hall_121/500180237/Mon_Jul_10_09:29:13_2023,## Combined Text ##\nNo readable text found in the images.,True,[pp],Smith_Hall_121,500180237,Mon_Jul_10_09:29:13_2023,1
2,data/Smith_Hall_121/500180237/Mon_Jul_10_09:39:25_2023,## Combined Text ##,False,[],Smith_Hall_121,500180237,Mon_Jul_10_09:39:25_2023,0
3,data/Smith_Hall_121/500180237/Mon_Jul_10_11:29:04_2023,## Combined Text ##\nNo readable text was found in the images.,False,[],Smith_Hall_121,500180237,Mon_Jul_10_11:29:04_2023,0
4,data/Smith_Hall_121/500180237/Mon_Jul_10_10:52:08_2023,## Combined Text ##,False,[],Smith_Hall_121,500180237,Mon_Jul_10_10:52:08_2023,0


## Filter and visualize Gemini results

Every chart below listens to the same attribute/category filters. Narrow the selection to understand which scenes were flagged and why, then inspect an individual scene at the bottom.


In [ ]:
attribute_widgets = {}
for attr in attribute_columns:
    values = sorted({str(v) for v in scenes_df[attr].dropna().unique()})
    options = ["All"] + values
    attribute_widgets[attr] = widgets.Dropdown(
        options=options,
        value="All",
        description=attr.replace("_", " ").title(),
        layout=widgets.Layout(width="280px"),
    )

category_widget = widgets.SelectMultiple(
    options=all_categories,
    value=tuple(all_categories) if all_categories else (),
    description="Flag categories",
    rows=min(max(len(all_categories), 3), 10) if all_categories else 3,
    layout=widgets.Layout(width="320px"),
    disabled=not all_categories,
)

flagged_widget = widgets.Dropdown(
    options=[
        ("Any flag state", "any"),
        ("Flagged only", "flagged"),
        ("Not flagged", "clean"),
    ],
    value="any",
    description="Flag state",
)

keyword_widget = widgets.Text(
    value="",
    description="Keyword",
    placeholder="Search response text...",
    continuous_update=False,
    layout=widgets.Layout(width="50%"),
)

rows_slider = widgets.IntSlider(
    min=5,
    max=40,
    step=5,
    value=10,
    description="Rows to preview",
    continuous_update=False,
)

if attribute_columns:
    group_attr_widget = widgets.Dropdown(
        options=attribute_columns,
        value=attribute_columns[0],
        description="Group attribute",
        layout=widgets.Layout(width="50%"),
    )
else:
    group_attr_widget = None

scene_selector = widgets.Dropdown(
    options=[("Select a scene", None)],
    value=None,
    description="Scene to inspect",
    layout=widgets.Layout(width="80%"),
    disabled=True,
)
scene_detail_output = widgets.Output()

scene_prev_button = widgets.Button(description="Previous", icon="arrow-left", disabled=True)
scene_next_button = widgets.Button(description="Next", icon="arrow-right", disabled=True)
scene_status = widgets.HTML("<em>Select a scene from the table below.</em>")


controls_children = [widgets.HTML("<b>Scene filters</b>")]
if attribute_widgets:
    attr_widgets = list(attribute_widgets.values())
    for start in range(0, len(attr_widgets), 2):
        controls_children.append(widgets.HBox(attr_widgets[start : start + 2]))
controls_children.append(widgets.HBox([category_widget, flagged_widget]))
controls_children.append(keyword_widget)
display(widgets.VBox(controls_children))

shared_controls = {f"attr__{attr}": widget for attr, widget in attribute_widgets.items()}
shared_controls["category_filter"] = category_widget
shared_controls["flagged_filter"] = flagged_widget
shared_controls["keyword_filter"] = keyword_widget



def _scene_selector_values():
    return [value for label, value in scene_selector.options if value]


def _update_scene_nav_state():
    values = _scene_selector_values()
    current = scene_selector.value
    if not values:
        scene_status.value = "<em>No scenes match the current filters.</em>"
        scene_prev_button.disabled = True
        scene_next_button.disabled = True
        return
    if current not in values:
        scene_status.value = "<em>Select a scene from the table below.</em>"
        scene_prev_button.disabled = True
        scene_next_button.disabled = True
        return
    idx = values.index(current)
    total = len(values)
    scene_status.value = f"Scene {idx + 1} of {total}"
    scene_prev_button.disabled = idx == 0
    scene_next_button.disabled = idx == total - 1


def _on_scene_prev(_):
    values = _scene_selector_values()
    if not values:
        return
    current = scene_selector.value
    if current not in values:
        scene_selector.value = values[0]
        return
    idx = values.index(current)
    if idx > 0:
        scene_selector.value = values[idx - 1]


def _on_scene_next(_):
    values = _scene_selector_values()
    if not values:
        return
    current = scene_selector.value
    if current not in values:
        scene_selector.value = values[0]
        return
    idx = values.index(current)
    if idx < len(values) - 1:
        scene_selector.value = values[idx + 1]

def _filter_scenes_from_kwargs(kwargs):
    data = scenes_df.copy()
    if data.empty:
        return data
    for attr in attribute_columns:
        selection = kwargs.get(f"attr__{attr}")
        if selection and selection != "All":
            data = data[data[attr].astype(str) == str(selection)]
    category_selection = kwargs.get("category_filter") or []
    if category_selection:
        selection_set = set(category_selection)
        data = data[data["flag_categories"].apply(lambda cats: bool(selection_set.intersection(cats)))]
    flag_filter = kwargs.get("flagged_filter", "any")
    if flag_filter == "flagged":
        data = data[data["is_flagged"]]
    elif flag_filter == "clean":
        data = data[~data["is_flagged"]]
    keyword = (kwargs.get("keyword_filter") or "").strip()
    if keyword:
        data = data[data["response_text"].str.contains(keyword, case=False, na=False)]
    return data


def update_scene_selector(scene_ids):
    sorted_ids = sorted({sid for sid in scene_ids if sid})
    options = [("Select a scene", None)] + [(sid, sid) for sid in sorted_ids]
    previous = scene_selector.value
    scene_selector.options = options
    scene_selector.disabled = len(options) <= 1
    if previous in sorted_ids:
        scene_selector.value = previous
    elif sorted_ids:
        scene_selector.value = sorted_ids[0]
    else:
        scene_selector.value = None
    _update_scene_nav_state()


def render_scene_summary(**kwargs):
    filtered = _filter_scenes_from_kwargs(kwargs)
    if filtered.empty:
        display(Markdown("No scenes match the current filters."))
        return
    total = len(filtered)
    flagged = int(filtered["is_flagged"].sum())
    summary = {
        "Scenes in selection": total,
        "Flagged scenes": flagged,
        "Flagged share": f"{flagged / total * 100:.1f}%" if total else "0%",
    }
    display(pd.DataFrame([summary]))


def render_category_chart(**kwargs):
    filtered = _filter_scenes_from_kwargs(kwargs)
    exploded = filtered["flag_categories"].explode()
    if exploded.empty:
        display(Markdown("No flag categories in the current selection."))
        return
    category_counts = (
        exploded.value_counts()
        .rename_axis("flag_category")
        .reset_index(name="count")
    )
    fig = px.bar(
        category_counts,
        x="flag_category",
        y="count",
        text_auto=".0f",
        title="Flagged categories",
    )
    fig.update_layout(xaxis_title="Category", yaxis_title="Scenes")
    fig.show()


def render_attribute_stack(group_attribute=None, **kwargs):
    if not group_attribute:
        display(Markdown("Choose an attribute to build the stacked chart."))
        return
    filtered = _filter_scenes_from_kwargs(kwargs)
    if filtered.empty:
        display(Markdown("No scenes to chart."))
        return
    if group_attribute not in filtered.columns:
        display(Markdown(f"{group_attribute} is not present in this data."))
        return
    exploded = filtered.explode("flag_categories")
    exploded = exploded[exploded["flag_categories"].notna()]
    if exploded.empty:
        display(Markdown("No flag categories to chart for this attribute."))
        return
    grouped = (
        exploded.groupby([group_attribute, "flag_categories"])
        .size()
        .reset_index(name="scenes")
    )
    fig = px.bar(
        grouped,
        x=group_attribute,
        y="scenes",
        color="flag_categories",
        text_auto=".0f",
        title=f"Flag categories by {group_attribute}",
    )
    fig.update_layout(barmode="stack", xaxis_title=group_attribute, yaxis_title="Scenes")
    fig.show()


def render_scene_table(rows_to_show=10, **kwargs):
    filtered = _filter_scenes_from_kwargs(kwargs)
    update_scene_selector(filtered["scene_path"].dropna().unique())
    if filtered.empty:
        display(Markdown("No scenes to preview."))
        return
    columns = ["scene_path", "is_flagged", "flag_categories"] + attribute_columns
    available_cols = [col for col in columns if col in filtered.columns]
    preview = filtered[available_cols].copy()
    preview["flag_categories"] = preview["flag_categories"].apply(
        lambda cats: ", ".join(cats) if cats else "None"
    )
    display(preview.head(rows_to_show))


def render_scene_detail(change=None):
    scene_id = scene_selector.value
    with scene_detail_output:
        scene_detail_output.clear_output()
        if not scene_id:
            display(Markdown("Select a scene above to inspect its Gemini response."))
            return
        match = scenes_df[scenes_df["scene_path"] == scene_id]
        if match.empty:
            display(Markdown("Scene not found in the loaded data."))
            return
        row = match.iloc[0]
        display(Markdown(f"**Scene:** `{scene_id}`"))
        if attribute_columns:
            details = ", ".join(
                f"{attr}: {row.get(attr)}" for attr in attribute_columns if pd.notna(row.get(attr))
            )
            if details:
                display(Markdown(f"*Attributes:* {details}"))
        categories = row.get("flag_categories") or []
        display(Markdown(f"*Flag categories:* {', '.join(categories) if categories else 'None'}"))
        text = row.get("response_text") or "(No text returned.)"
        safe_text = html.escape(str(text))
        display(Markdown("**Response text**"))
        display(Markdown(f"<pre style='white-space:pre-wrap'>{safe_text}</pre>"))
        images = scene_image_map.get(scene_id, [])
        if images:
            display(Markdown("**Sample images**"))
            for image_path in images[:4]:
                absolute = (PROJECT_ROOT / image_path).resolve()
                display(Markdown(f"`{image_path}`"))
                if absolute.exists():
                    display(IPImage(filename=str(absolute), width=320))
                else:
                    display(Markdown("(Image not found locally)"))
        else:
            display(Markdown("_No indexed images were found for this scene._"))



def _handle_scene_change(change):
    _update_scene_nav_state()
    render_scene_detail(change)


scene_selector.observe(_handle_scene_change, names="value")

scene_prev_button.on_click(_on_scene_prev)
scene_next_button.on_click(_on_scene_next)

scene_summary_panel = widgets.interactive_output(render_scene_summary, dict(shared_controls))
category_panel = widgets.interactive_output(render_category_chart, dict(shared_controls))
table_controls = dict(shared_controls)
table_controls["rows_to_show"] = rows_slider
table_panel = widgets.interactive_output(render_scene_table, table_controls)


display(Markdown("### Scene summary"))
display(scene_summary_panel)


display(Markdown("### Flag categories"))
display(category_panel)

if group_attr_widget:
    group_controls = dict(shared_controls)
    group_controls["group_attribute"] = group_attr_widget
    attribute_panel = widgets.interactive_output(render_attribute_stack, group_controls)
    display(Markdown("### Flag categories by attribute"))
    display(group_attr_widget)
    display(attribute_panel)


display(Markdown("### Scene table"))
display(rows_slider)
display(table_panel)


display(Markdown("### Scene detail"))
scene_nav_row = widgets.HBox([
    scene_prev_button,
    scene_selector,
    scene_next_button,
])
display(scene_status)
display(scene_nav_row)
display(scene_detail_output)

render_scene_detail()


### Scene summary

Output()

### Flag categories

Output()

### Flag categories by attribute

...

IntSlider(value=10, continuous_update=False, description='Rows to preview', max=40, min=5, step=5)

Output()

### Scene detail

HTML(value='Scene 1 of 2')

Output()